# Étape 3 — Construire le RAG multimodal avec Ollama

Exécute d'abord la cellule de configuration ci-dessous pour reconnecter les collections Chroma et charger les modèles de recherche depuis le cache local. Ollama doit être démarré localement (API sur 127.0.0.1:11434). LangGraph choisit les modalités, lance une recherche complémentaire si les résultats sont faibles et fournit les sources à Ollama. Aucun modèle n'est téléchargé ici : indique dans le widget le nom exact d'un modèle déjà installé dans Ollama.

In [1]:
from pathlib import Path
import os

racine_projet = next(
    (
        dossier
        for dossier in (Path.cwd(), *Path.cwd().parents)
        if (dossier / "pyproject.toml").is_file()
        and (dossier / "chroma_db").is_dir()
    ),
    None,
)
if racine_projet is None:
    raise FileNotFoundError(
        "Racine du projet introuvable : pyproject.toml et chroma_db/ "
        "doivent être présents."
    )
os.chdir(racine_projet)


import chromadb
import torch
import torch.nn.functional as F
from IPython.display import Markdown, display
from PIL import Image
from transformers import CLIPModel, CLIPProcessor

# Reconnecter les collections persistantes et charger les modèles depuis le cache local.
client_chroma = chromadb.PersistentClient(path="./chroma_db")
collection_images = client_chroma.get_collection("manual_images_clip")
collection_texte = client_chroma.get_collection("manual_text_tables")

modele_embeddings = None

nom_modele_clip = "openai/clip-vit-base-patch32"
processor_clip = CLIPProcessor.from_pretrained(
    nom_modele_clip, local_files_only=True
)
modele_clip = CLIPModel.from_pretrained(
    nom_modele_clip, local_files_only=True
)
modele_clip.eval()
print("Modèle CLIP prêt.")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Modèle CLIP prêt.


In [2]:
import sys
from langchain_ollama import ChatOllama

if str(racine_projet) not in sys.path:
    sys.path.insert(0, str(racine_projet))

from partie2.rag_runtime import (
    Modalite,
    RAGRuntime,
    choisir_modalites,
    normaliser_mots,
)

rag_runtime = RAGRuntime(
    collection_images=collection_images,
    collection_texte=collection_texte,
    modele_embeddings=modele_embeddings,
    processor_clip=processor_clip,
    modele_clip=modele_clip,
    torch_module=torch,
    functional_module=F,
)
rechercher = rag_runtime.rechercher
rechercher_sources = rag_runtime.rechercher_sources
formater_sources = rag_runtime.formater_sources
creer_agent_rag = rag_runtime.creer_agent_rag
creer_agent_react = rag_runtime.creer_agent_react
retriever_tool = rag_runtime.creer_retriever_tool()

print("RAG multimodal prêt : texte, images, outil retriever_tool.")
print("Outil :", retriever_tool.name, "—", retriever_tool.description[:80])

RAG multimodal prêt : texte, images, outil retriever_tool.
Outil : retriever_tool — Recherche multimodale dans le document indexé (texte et images). À utiliser avan


In [3]:
# Tests fonctionnels sans appel au modèle Ollama.
from types import SimpleNamespace

assert choisir_modalites("Quelle température indique le tableau ?") == ["text", "table"]
assert choisir_modalites("Montre-moi le schéma de la machine") == ["text", "image"]

source_test = {
    "id": "table-test",
    "contenu": "We propose the Transformer, based solely on attention mechanisms.",
    "metadata": {
        "source": "attention is all you need.pdf",
        "page_number": 60,
        "content_type": "table",
    },
    "distance": 0.2,
}
contexte_test, citations_test = formater_sources([source_test])
assert "[1] page 60 (table)" in contexte_test
assert citations_test[0]["document"] == "attention is all you need.pdf"
assert retriever_tool.name == "retriever_tool"

# Vérifier qu'une recherche complémentaire trouve l'image si le texte est faible.
def recherche_simulee(question, modalite):
    if modalite == "text":
        return [{**source_test, "distance": 1.2}]
    if modalite == "image":
        return [{**source_test, "id": "image-test", "distance": 0.3}]
    return []

sources_complementaires = rag_runtime.rechercher_sources(
    "question visuelle", ["text"], fonction_recherche=recherche_simulee
)
assert [source["id"] for source in sources_complementaires] == ["image-test"]

class FauxLLM:
    def invoke(self, messages):
        assert "Transformer" in messages[1].content
        assert "## Réponse" in messages[0].content
        assert "## Éléments du manuel" in messages[0].content
        return SimpleNamespace(content=(
            "## Réponse\nLe Transformer repose uniquement sur l'attention [1].\n\n"
            "## Éléments du manuel\n- Attention mechanisms [1]."
        ))

etapes_test = []
agent_test = creer_agent_rag(
    FauxLLM(),
    fonction_recherche=lambda question, modalites: [source_test],
    notifier_etape=etapes_test.append,
)
resultat_test = agent_test.invoke({"question": "Quelle température ?"})
assert "[1]" in resultat_test["reponse"]
assert resultat_test["sources"][0]["id"] == "table-test"
assert len(etapes_test) == 2
assert "Recherche en cours" in etapes_test[0]
assert "Génération de la réponse" in etapes_test[1]
print("Tests réussis : routage, recherche complémentaire, citations et orchestration LangGraph (LLM simulé).")

Tests réussis : routage, recherche complémentaire, citations et orchestration LangGraph (LLM simulé).


# Étapes 5 et 6 — retriever_tool et agent ReAct (LangGraph)

`retriever_tool` encapsule la recherche CLIP. `create_react_agent` enchaîne Reasoning → Action (outil) → Observation. Ollama `qwen2.5:3b` doit tourner.


In [ ]:
print("Nom :", retriever_tool.name)
print("Description :", retriever_tool.description)
observation = retriever_tool.invoke({"query": "What is scaled dot-product attention?"})
print("\nObservation retriever_tool :\n")
print(observation[:1500])

print("\n--- Agent ReAct (1 question) ---")
agent_react = creer_agent_react(ChatOllama(model="qwen2.5:3b", temperature=0))
etat_react = agent_react.invoke({
    "messages": [("user", "What does the encoder-decoder figure show? Use retriever_tool.")]
})
for msg in etat_react["messages"]:
    print(f"\n[{type(msg).__name__}]")
    print(str(getattr(msg, "content", ""))[:1200])


In [4]:
# Validation de bout en bout avec Ollama et les collections Chroma réelles.
from time import perf_counter

questions_validation = [
    "What is scaled dot-product attention?",
    "What is multi-head attention in the Transformer?",
    "What does the encoder-decoder figure show?",
]

for question_validation in questions_validation:
    print(f"\nQuestion : {question_validation}")
    debut_validation = perf_counter()
    agent_validation = creer_agent_rag(
        ChatOllama(model="qwen2.5:3b", temperature=0)
    )
    resultat_validation = agent_validation.invoke(
        {"question": question_validation}
    )
    _, citations_validation = formater_sources(
        resultat_validation.get("sources", [])
    )
    print("\nRéponse :")
    display(Markdown(resultat_validation["reponse"]))
    print("Sources :")
    if citations_validation:
        for citation_validation in citations_validation:
            print(
                f"- page {citation_validation['page']} "
                f"({citation_validation['type']}, "
                f"distance {citation_validation['distance']:.3f}) : "
                f"{citation_validation['contenu'][:350]}"
            )
    else:
        print("Aucune source retenue.")
    print(f"Durée : {perf_counter() - debut_validation:.1f} s")


Question : What is scaled dot-product attention?

Réponse :


## Réponse
Scaled dot-product attention est une version améliorée de l'attention basée sur le produit scalaire qui introduit un facteur de scalage pour améliorer la stabilité et l'efficacité.

## Éléments du manuel
[1] page 4 (text): plicative) attention. Dot-product attention is identical to our algorithm, except for the scaling factor

[2] page 1 (text): Noam proposed scaled dot-product attention, multi-head
attention and the parameter-free position representation and became the other person involved in nearly every

## Limites
La source ne fournit pas d'informations sur la valeur exacte du facteur de scalage utilisé dans l'attention à produit scalaire échelonné.

Sources :
- page 4 (text, distance 0.100) : plicative) attention. Dot-product attention is identical to our algorithm, except for the scaling factor
- page 1 (text, distance 0.139) : has been crucially involved in every aspect of this work. Noam proposed scaled dot-product attention, multi-head
attention and the parameter-free position representation and became the other person involved in nearly every
- page 2 (text, distance 0.156) : described in section 3.2.
Self-attention, sometimes called intra-attention is an attention mechanism relating different positions
of a single sequence in order to compute a representation of the sequence. Self-attention has been
- page 9 (text, distance 0.176) : suggests that determining compatibility is not easy and that a more sophisticated compatibility
function than dot product may be beneﬁcial. We further observe in rows (C) and (D) that, as expected,
Durée : 74.3 s

Question : What is multi-head attention in the Transformer?

Réponse :


## Réponse
Multi-head attention in the Transformer is used to process multiple queries simultaneously, allowing it to capture different aspects or perspectives of the input data more effectively.

## Éléments du manuel
[3] 3.2.3 Applications of Attention in our Model : Le modèle Transformer utilise l'attention multi-têtes de trois façons différentes :
• Dans les couches d'attention "encoder-decoder", les requêtes proviennent des précédentes couches de décodage.

## Limites
La source ne fournit pas d'exemples spécifiques ou de détails techniques sur comment fonctionne l'attention multi-têtes dans le Transformer.

Sources :
- page 2 (text, distance 0.134) : [9], consuming the previously generated symbols as additional input when generating the next.
The Transformer follows this overall architecture using stacked self-attention and point-wise, fully
- page 9 (text, distance 0.151) : plan to extend the Transformer to problems involving input and output modalities other than text and
to investigate local, restricted attention mechanisms to efﬁciently handle large inputs and outputs
- page 5 (text, distance 0.168) : 3.2.3 Applications of Attention in our Model
The Transformer uses multi-head attention in three different ways:
• In "encoder-decoder attention" layers, the queries come from the previous decoder layer,
- page 2 (text, distance 0.170) : The Transformer allows for signiﬁcantly more parallelization and can reach a new state of the art in
translation quality after being trained for as little as twelve hours on eight P100 GPUs.
2 Background
Durée : 37.4 s

Question : What does the encoder-de

## Réponse
The encoder-decoder figure shows the architecture of a neural network model, specifically highlighting the structure and operations involved in both the encoder and decoder layers. It illustrates how each position in the decoder can attend to all positions in the input sequence through multi-head attention mechanisms, which are derived from the output of the encoder.

## Éléments du manuel
- [1] page 5 (text): Encoder: Each position in the encoder can attend to all positions in the previous layer of the encoder. Similarly, self-attention layers in the decoder allow each position in the decoder to attend to.
- [2] page 5 (text): and the memory keys and values come from the output of the encoder. This allows every position in the decoder to attend over all positions in the input sequence. This mimics the typical encoder-decoder attention mechanisms in sequence-to-sequence models such as [31, 2, 8].
- [3] page 3 (text): Decoder: The decoder is also composed of a stack of N = 6 identical layers. In addition to the two sub-layers in each encoder layer, the decoder inserts a third sub-layer, which performs multi-head attention over the output of the encoder stack.
- [4] page 3 (text): Similar to the encoder, we employ residual connections around each of the sub-layers, followed by layer normalization. We also modify the self-attention layers in the decoder.
- [5] page 4 (image): Cette image représente une structure de réseau neuronal, probablement d'un modèle de reconnaissance de langage.
- [6] page 4 (image): La figure montre un schéma d'un processus de calcul en machine learning, spécifiquement impliqué dans la compréhension de la matrice de produit de symbole (MatMul).
- [7] page 3 (image): La figure montre un schéma de réseau de neurones pour une machine à l'apprentissage automatique. Les éléments principaux sont les suivants :

1. **Input Embedding**: Un ensemble de vecteurs de positionnement qui sont ajoutés et normalisés avant d'être utilisés pour l'entraînement.
2. **Multi-Head Attention**: Un type de réseau de neurones qui utilise des entités de positionnement pour calculer des combinaisons de vecteurs. Ces combinaisons sont ensuite normalisées et ajoutées à l'entraînement.

## Limites
La figure ne fournit pas d'informations sur la façon dont les positions dans le decoder interagissent avec celles du encoder ou comment elles utilisent les valeurs et les clés de mémoire.

Sources :
- page 5 (text, distance 0.119) : encoder. Each position in the encoder can attend to all positions in the previous layer of the
encoder.
• Similarly, self-attention layers in the decoder allow each position in the decoder to attend to
- page 5 (text, distance 0.132) : and the memory keys and values come from the output of the encoder. This allows every
position in the decoder to attend over all positions in the input sequence. This mimics the
typical encoder-decoder attention mechanisms in sequence-to-sequence models such as
[31, 2, 8].
- page 3 (text, distance 0.136) : Decoder: The decoder is also composed of a stack ofN = 6 identical layers. In addition to the two
sub-layers in each encoder layer, the decoder inserts a third sub-layer, which performs multi-head
- page 3 (text, distance 0.167) : attention over the output of the encoder stack. Similar to the encoder, we employ residual connections
around each of the sub-layers, followed by layer normalization. We also modify

In [7]:
import ipywidgets as widgets
from IPython.display import clear_output

champ_modele_ollama = widgets.Text(
    value=os.getenv("OLLAMA_MODEL", "qwen2.5:3b"),
    placeholder="qwen2.5:3b",
    description="Modèle :",
    layout=widgets.Layout(width="80%"),
)
champ_question = widgets.Text(
    placeholder="Pose une question sur le manuel…",
    description="Question :",
    layout=widgets.Layout(width="100%"),
)
bouton_question = widgets.Button(description="Rechercher et répondre", button_style="primary")
sortie_rag = widgets.Output()


def afficher_progression(message: str) -> None:
    clear_output(wait=True)
    display(Markdown(f"*{message}*"))


def traiter_question(_):
    with sortie_rag:
        clear_output(wait=True)
        modele = champ_modele_ollama.value.strip().replace(" ", "")
        if modele.lower() in {"qwen2.5", "qwen2.5:3", "qwen25:3b"}:
            modele = "qwen2.5:3b"
        question = champ_question.value.strip()
        if not modele:
            print("Indique un modèle déjà installé dans Ollama, par exemple qwen2.5:3b.")
            return
        print(f"Modèle Ollama : {modele}")
        if not question:
            print("Saisis une question avant de lancer la recherche.")
            return
        try:
            agent = creer_agent_rag(
                ChatOllama(model=modele, temperature=0),
                notifier_etape=afficher_progression,
            )
            resultat = agent.invoke({"question": question})
            clear_output(wait=True)
            display(Markdown(resultat["reponse"]))
            _, citations = formater_sources(resultat.get("sources", []))
            if citations:
                display(Markdown("### Sources"))
                for citation in citations:
                    display(Markdown(
                        f"**[{citation['numero']}]** {citation['document']} — "
                        f"page {citation['page']} — {citation['type']} "
                        f"(distance {citation['distance']:.3f})\n\n"
                        f"{citation['contenu']}"
                    ))
                    chemin = citation.get("image_path")
                    if chemin and Path(chemin).is_file():
                        display(Image.open(chemin))
        except Exception as exc:
            print(f"Erreur RAG/Ollama : {type(exc).__name__}: {exc}")


bouton_question.on_click(traiter_question)
display(widgets.VBox([
    champ_modele_ollama,
    champ_question,
    bouton_question,
    sortie_rag,
]))